# 8. Visualize transformed data as heatmaps

Reads **transformed** parquet files from `work_dir/transformed/<band>/transformed_<YYYYMMDD>.parquet` and plots **airtime utilization (AU) heatmaps** **per class** (band).

- **Dropdown:** pick **date**; then pick **class** (frequency band, e.g. 195MHz, 539MHz).
- **X axis:** frequency (GHz)  
- **Y axis:** hour of day (00–23)  
- **Color:** AU % for that date and class (band); all thresholds for that class shown in subplots.

In [73]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

In [74]:
# Path to transformed data
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
transformed_dir = work_dir / "transformed"
if not transformed_dir.exists():
    raise FileNotFoundError(f"Transformed dir not found: {transformed_dir}")

# Discover available dates (YYYYMMDD) and classes (bands) from transformed_*.parquet
dates_found = set()
classes_found = []
for band_dir in sorted(transformed_dir.iterdir()):
    if not band_dir.is_dir():
        continue
    classes_found.append(band_dir.name)
    for p in band_dir.glob("transformed_*.parquet"):
        stem = p.stem  # transformed_20260203
        if stem.startswith("transformed_"):
            dates_found.add(stem.replace("transformed_", ""))
date_options = sorted(dates_found)
if not date_options:
    raise FileNotFoundError(f"No transformed_*.parquet files under {transformed_dir}")
# Format for display: YYYY-MM-DD
date_display = [f"{d[:4]}-{d[4:6]}-{d[6:8]}" for d in date_options]
class_options = sorted(classes_found)

# Dropdowns: date and class (band)
date_dropdown = widgets.Dropdown(
    options=list(zip(date_display, date_options)),
    value=date_options[0],
    description="Date:",
    style={"description_width": "50px"},
)
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=class_options[0] if class_options else None,
    description="Class:",
    style={"description_width": "50px"},
)
out = widgets.Output()

### Raw transformed data (for the first available date & class)

Run this cell to inspect the dataframe used for the heatmaps. Change the date/class in the dropdowns below and re-run the visualization to see that date/class; re-run this cell with different `sample_date` / `sample_class` to see their raw data.

In [75]:
# Pick which date and class to inspect (same format as dropdowns)
sample_date = date_options[0]       # YYYYMMDD, or set e.g. "20260203"
sample_class = class_options[0]      # e.g. "195MHz"

path = transformed_dir / sample_class / f"transformed_{sample_date}.parquet"
if not path.exists():
    print(f"File not found: {path}")
else:
    df_raw = pd.read_parquet(path)
    print("Shape:", df_raw.shape)
    print("Columns:", list(df_raw.columns))
    print("\nDtypes:\n", df_raw.dtypes)
    print("\n--- First 80 rows ---")
    display(df_raw.head(80))
    print("\n--- Summary: hour, threshold_dbm, au_pct ---")
    print("Unique hours:", sorted(df_raw["hour"].unique().tolist()))
    print("Unique threshold_dbm:", sorted(df_raw["threshold_dbm"].unique().tolist()))
    print("au_pct: min={:.4f}, max={:.4f}, mean={:.4f}".format(
        df_raw["au_pct"].min(), df_raw["au_pct"].max(), df_raw["au_pct"].mean()))
    print("freq_center_ghz: min={:.6f}, max={:.6f}".format(
        df_raw["freq_center_ghz"].min(), df_raw["freq_center_ghz"].max()))

Shape: (304, 6)
Columns: ['date', 'hour', 'band', 'freq_center_ghz', 'threshold_dbm', 'au_pct']

Dtypes:
 date                   str
hour                 int64
band                   str
freq_center_ghz    float64
threshold_dbm        int64
au_pct             float64
dtype: object

--- First 80 rows ---


,date,hour,band,freq_center_ghz,threshold_dbm,au_pct
0,2026-01-31,19,195MHz,0.17525,-110,3.262956
1,2026-01-31,19,195MHz,0.17575,-110,47.600768
2,2026-01-31,19,195MHz,0.17625,-110,0.000000
3,2026-01-31,19,195MHz,0.17675,-110,0.000000
4,2026-01-31,19,195MHz,0.17725,-110,0.000000
...,...,...,...,...,...,...
75,2026-01-31,20,195MHz,0.18275,-110,0.000000
76,2026-01-31,20,195MHz,0.18325,-110,0.000000
77,2026-01-31,20,195MHz,0.18375,-110,0.000000
78,2026-01-31,20,195MHz,0.18425,-110,0.000000



--- Summary: hour, threshold_dbm, au_pct ---
Unique hours: [19, 20, 21, 22, 23]
Unique threshold_dbm: [-110]
au_pct: min=0.0000, max=100.0000, mean=22.8000
freq_center_ghz: min=0.174250, max=0.207750


In [76]:
def build_heatmap_matrix(df_sub: pd.DataFrame) -> tuple[np.ndarray, list, list]:
    """Build (hours x freqs) matrix and ordered hour/freq labels from transformed rows."""
    hours = sorted(df_sub["hour"].unique())
    freqs = sorted(df_sub["freq_center_ghz"].unique())
    if not hours or not freqs:
        return np.zeros((0, 0)), [], []
    pivot = df_sub.pivot_table(
        index="hour", columns="freq_center_ghz", values="au_pct", aggfunc="first"
    ).reindex(index=hours, columns=freqs)
    # No row in transformed data = 0% AU (transform only writes rows where there was >=1 detection)
    matrix = np.nan_to_num(pivot.values, nan=0.0)
    return matrix, [f"{h:02d}:00" for h in hours], freqs


def plot_one_heatmap(matrix: np.ndarray, hour_labels: list, freq_ghz: list, title: str):
    """Plot a single AU heatmap with Plotly."""
    vmax = 100.0  # AU is percentage (0-100%)
    fig = go.Figure(data=go.Heatmap(
        x=freq_ghz,
        y=hour_labels,
        z=matrix,
        colorscale="Viridis",
        zmin=0,
        zmax=vmax,
        colorbar=dict(title="AU (%)"),
        hovertemplate="Freq: %{x:.4f} GHz<br>Time: %{y}<br>AU: %{z:.2f}%<extra></extra>",
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Freq (GHz)",
        yaxis_title="Hour",
        height=500,
        yaxis=dict(autorange="reversed"),
    )
    return fig

In [77]:
def update_heatmaps(date_yyyymmdd, class_band):
    with out:
        clear_output(wait=True)
        parquet_path = transformed_dir / class_band / f"transformed_{date_yyyymmdd}.parquet"
        date_dash = f"{date_yyyymmdd[:4]}-{date_yyyymmdd[4:6]}-{date_yyyymmdd[6:8]}"
        if not parquet_path.exists():
            print(f"No data: {parquet_path}")
            return
        df = pd.read_parquet(parquet_path)
        thresholds = sorted(df["threshold_dbm"].unique().tolist())
        if not thresholds:
            print("No threshold data in this file.")
            return
        n_th = len(thresholds)
        n_cols = min(3, n_th)
        n_rows = (n_th + n_cols - 1) // n_cols
        vmax_global = 0.0
        for th in thresholds:
            sub = df[df["threshold_dbm"] == th]
            matrix, _, _ = build_heatmap_matrix(sub)
            if matrix.size > 0:
                v = np.nanmax(matrix)
                if not np.isnan(v):
                    vmax_global = max(vmax_global, v)
        zmax = 100.0  # AU is percentage (0-100%)
        fig = make_subplots(
            rows=n_rows, cols=n_cols,
            subplot_titles=[f"{th} dBm" for th in thresholds],
            vertical_spacing=0.12, horizontal_spacing=0.08,
        )
        for idx, th in enumerate(thresholds):
            sub = df[df["threshold_dbm"] == th]
            matrix, hour_labels, freq_ghz = build_heatmap_matrix(sub)
            row, col = idx // n_cols + 1, idx % n_cols + 1
            fig.add_trace(
                go.Heatmap(
                    x=freq_ghz, y=hour_labels, z=matrix,
                    colorscale="Viridis", zmin=0, zmax=zmax,
                    showscale=(col == n_cols),
                ), row=row, col=col,
            )
        fig.update_traces(zmax=zmax, zmin=0)
        fig.update_layout(
            title=f"AU (%) — {date_dash} — {class_band} (per class)",
            height=max(520, 480 * n_rows),
        )
        for i in range(1, n_rows + 1):
            for j in range(1, n_cols + 1):
                fig.update_xaxes(title_text="Freq (GHz)", row=i, col=j)
                fig.update_yaxes(title_text="Hour", row=i, col=j, autorange="reversed")
        fig.show()

widgets.interactive_output(update_heatmaps, {"date_yyyymmdd": date_dropdown, "class_band": class_dropdown})
display(widgets.HBox([date_dropdown, class_dropdown]), out)
# Trigger initial render
update_heatmaps(date_dropdown.value, class_dropdown.value)

Output()

The plot above shows all power thresholds for the selected date and class in one figure (one subplot per threshold).

In [78]:
# Use the Date and Class dropdowns above to view heatmaps per class.